# **DIABETES PREDICTION OF 2023**

## **0. Library import**

In [1]:
import os
import sys

# Add the root path into the python path
root_path = os.path.abspath(os.path.join(".."))
if not root_path in sys.path:
    sys.path.insert(0, root_path)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import LOG_DIR, RAW_DATA_DIR, TRAINING_DIR, SCALERS_DIR, MODEL_DIR
from src.utils import load_data
from src.data import BrfssDataLoader, BrfssDataCleaner
from src.visualization import (
    plot_feature_distributions_comparison, 
    plot_diabetes_distribution_by_indicators, 
    plot_correlation_matrix, 
    plot_class_distribution,
)
from src.features import DiabetesFeatureEngineering
from src.models import DiabetesLogisticRegression, DiabetesRandomForest

## **1. Download and load dataset**

In [3]:
# Download and extract BRFSS 2023 dataset
brfss_dataloader = BrfssDataLoader(log_file=f"{LOG_DIR}/6_brfss_data_loader.log")
brfss_dataloader.load_data(
    urls=["https://www.cdc.gov/brfss/annual_data/2023/files/LLCP2023XPT.zip"],
    filenames=["brfss_2023.XPT"]
)

2025-09-14 23:53:23,507 - [src.data] - INFO - BrfssDataLoader initialized successfully
2025-09-14 23:53:23,508 - [src.data] - INFO - Starting data loading process...
2025-09-14 23:53:23,509 - [src.data] - INFO - Processing: LLCP2023XPT.zip
2025-09-14 23:53:23,509 - [src.data] - INFO - LLCP2023XPT.zip not found locally. Downloading...
2025-09-14 23:53:23,510 - [src.data] - INFO - Navigating to URL: https://www.cdc.gov/brfss/annual_data/2023/files/LLCP2023XPT.zip
2025-09-14 23:53:23,641 - [src.data] - INFO - Starting download: LLCP2023XPT.zip (91052.56 KB)
LLCP2023XPT.zip: 100%|██████████| 88.9M/88.9M [00:03<00:00, 25.4MB/s]
2025-09-14 23:53:27,316 - [src.data] - INFO - Downloaded: LLCP2023XPT.zip -> /home/qctrung/Projects/machine-learning/diabetes-prediction-and-data-balancing/data/raw/zip/LLCP2023XPT.zip
2025-09-14 23:53:27,317 - [src.data] - INFO - Attempting to extract zip file: /home/qctrung/Projects/machine-learning/diabetes-prediction-and-data-balancing/data/raw/zip/LLCP2023XPT.zi

In [4]:
# Filter necessary features
brfss_data_cleaner = BrfssDataCleaner(log_file=f"{LOG_DIR}/6_brfss_data_cleaner.log")
cleaned_df = brfss_data_cleaner.clean(file_path=f"{RAW_DATA_DIR}/brfss_2023.XPT", dropna=True)

2025-09-14 23:53:29,934 - [src.data] - INFO - BrfssDataCleaner initialized successfully
2025-09-14 23:53:29,935 - [src.data] - INFO - Selecting features from /home/qctrung/Projects/machine-learning/diabetes-prediction-and-data-balancing/data/raw/brfss_2023.XPT
2025-09-14 23:53:52,501 - [src.data] - INFO - Processing feature values for training
2025-09-14 23:53:53,331 - [src.data] - INFO - Feature processing completed. Shape: (250090, 20)


## **2. Data visualization**

In [ ]:
# Distribution of BMI by Diabetes status in 2023
plot_feature_distributions_comparison(
    df_list=[cleaned_df],
    year_list=[2023],
    feature="BMI",
)

In [ ]:
# Distribution of Age by Diabetes status in 2023
plot_feature_distributions_comparison(
    df_list=[cleaned_df],
    year_list=[2023],
    feature="Age"
)

In [ ]:
# Distribution of General Health by Diabetes status in 2023
plot_feature_distributions_comparison(
    df_list=[cleaned_df],
    year_list=[2023],
    feature="GenHlth"
)

In [ ]:
# Draw boxplot to identify outlier
fig, ax = plt.subplots(1, 3, figsize=(16, 9))

sns.boxplot(data=cleaned_df, y='BMI', ax=ax[0])
ax[0].set_title('Interquartile range on BMI')

sns.boxplot(data=cleaned_df, y='MentHlth', ax=ax[1])
ax[1].set_title('Interquartile range on MentHlth')

sns.boxplot(data=cleaned_df, y='PhysHlth', ax=ax[2])
ax[2].set_title('Interquartile range on PhysHlth')

plt.suptitle(t="Boxplot analysis of continuous features (BMI, MentHlth, PhysHlth)", fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 9))

sns.boxplot(data=cleaned_df, y='BMI', hue='Diabetes', ax=ax[0])
ax[0].set_title('Interquartile range of BMI visualized by Diabetes')

sns.boxplot(data=cleaned_df, y='MentHlth', hue='Diabetes', ax=ax[1])
ax[1].set_title('Interquartile range of MentHlth viusalized by Diabetes')

sns.boxplot(data=cleaned_df, y='PhysHlth', hue='Diabetes', ax=ax[2])
ax[2].set_title('Interquartile range of PhysHlth viusalized by Diabetes')

plt.suptitle(t="Distribution of numerical features by diabete status", fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of diabetes by physical health indicators in 2023
plot_diabetes_distribution_by_indicators(
    df=cleaned_df,
    indicators=["BMI", "HighBP", "HighChol", "DiffWalk"],
    year=2023,
    title="Distribution of diabetes by physical health indicators"
)

In [ ]:
# Distribution of diabetes by demographics and lifestyle indicators in 2023
plot_diabetes_distribution_by_indicators(
    df=cleaned_df,
    indicators=["Age", "GenHlth", "Sex", "HvyAlcoholConsump"],
    year=2023,
    title="Distribution of diabetes by demographics and lifestyle indicators"
)

In [ ]:
# Distribution of diabetes by mental and general health indicators in 2023
plot_diabetes_distribution_by_indicators(
    df=cleaned_df,
    indicators=["MentHlth", "PhysHlth", "GenHlth", "NoDocbcCost"],
    year=2023,
    title="Distribution of diabetes by mental and general health indicators"
)


In [ ]:
# Distribution of diabetes by behavioral health indicators in 2023
plot_diabetes_distribution_by_indicators(
    df=cleaned_df,
    indicators=["Smoker", "HeartDiseaseorAttack", "PhysActivity"],
    year=2023,
    title="Distribution of diabetes by behavioral health indicators",
    figsize=(18, 8)
)

In [ ]:
# Display correlation heatmap
plot_correlation_matrix(
    df=cleaned_df,
    title="Correlation matrix of all features"
)

In [ ]:
# Plot distribution of classes in the dataset
plot_class_distribution(
    y=cleaned_df["Diabetes"],
    title="Distribution of Diabetes"
)

In [ ]:
diabetes_mapping = {
    0: "No",
    1: "Pre-diabetes",
    2: "Diabetes"
}
cleaned_df['Diabetes_label'] = cleaned_df['Diabetes'].map(diabetes_mapping)
# General Health distribution
sns.countplot(data=cleaned_df, x="Diabetes", hue="Diabetes_label")
plt.title('Diabetes distribution in 2023')
plt.xlabel('Diabetes')

plt.tight_layout()
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## **3. Feature engineering**

In [ ]:
# Get selected features and encoders of 2017, 2019, 2021
selected_features = load_data(f"{TRAINING_DIR}/selected_features.pkl")
encoders = load_data(f"{TRAINING_DIR}/encoders.pkl")

# Apply a complete feature engineering pipeline
diabetes_feature_engineering = DiabetesFeatureEngineering(log_file=f"{LOG_DIR}/6_data_feature_engineering.log")
processed_df = diabetes_feature_engineering.process_for_inference(
    df=cleaned_df,
    selected_features=selected_features,
    encoders=encoders
)

In [ ]:
# Display the first five rows in the dataset
processed_df.head()

In [ ]:
# Display overall about dataset
processed_df.info()

## **4. Data preparation**

In [ ]:
# Separate features (X) and target variable (y) for model training
X, y = processed_df.drop("Diabetes", axis=1), processed_df["Diabetes"]

In [2]:
# Load pretrained Min-Max scaler
tomek_scaler = load_data(f"{SCALERS_DIR}/tomek_scaler.pkl")
smote_enn_scaler = load_data(f"{SCALERS_DIR}/smote_enn_scaler.pkl")
rus_scaler = load_data(f"{SCALERS_DIR}/rus_scaler.pkl")

NameError: name 'load_data' is not defined

In [ ]:
# Normalize features
tomek_X = tomek_scaler.transform(X)
smote_enn_X = smote_enn_scaler.transform(X)
rus_X = rus_scaler.transform(X)